# 27. Vision Transformer ViT 핵심 아이디어

이 노트북은 `26_Attention_기초_복습.ipynb` 다음 단계로, Vision Transformer가 이미지를 어떻게 Transformer 입력으로 바꾸고 분류를 수행하는지 이해합니다.

ViT의 핵심은 이미지를 작은 patch로 자른 뒤 각 patch를 token처럼 다루는 것입니다. 그 앞에 class token을 붙이고 positional embedding을 더한 다음 Transformer encoder에 넣습니다.

이번 노트북의 목표는 다음과 같습니다.

- patch embedding의 입력과 출력 shape을 이해합니다.
- class token과 positional embedding의 역할을 이해합니다.
- Transformer encoder를 거친 뒤 class token으로 분류하는 흐름을 파악합니다.
- CNN 기반 분류 모델과 ViT 기반 분류 모델의 차이를 정리합니다.

## 27-1. 준비

작은 예제 이미지와 NumPy 행렬을 사용해 ViT의 입력 변환 과정을 직접 확인합니다. 여기서는 학습 가능한 파라미터를 임의 값으로 두고 shape과 흐름에 집중합니다.

In [ ]:
import numpy as np
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

available_fonts = {f.name for f in fm.fontManager.ttflist}
for font_name in ['Malgun Gothic', 'AppleGothic', 'NanumGothic']:
    if font_name in available_fonts:
        plt.rcParams['font.family'] = font_name
        break

plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.unicode_minus'] = False
np.set_printoptions(precision=3, suppress=True)
np.random.seed(42)

## 27-2. ViT의 전체 흐름

ViT는 이미지를 다음 순서로 처리합니다.

```text
image
  -> split into patches
  -> flatten each patch
  -> linear projection to patch embeddings
  -> prepend class token
  -> add positional embeddings
  -> Transformer encoder
  -> classification head using class token
```

CNN처럼 feature map을 점진적으로 줄이는 구조가 아니라, patch token sequence를 Transformer encoder에 넣는 구조입니다.

In [ ]:
image = np.zeros((8, 8, 3))
image[:4, :4, 0] = 0.9
image[:4, 4:, 1] = 0.8
image[4:, :4, 2] = 0.8
image[4:, 4:, :] = [0.8, 0.6, 0.1]

patch_size = 2
fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(image)
ax.set_title('8 x 8 예제 이미지와 2 x 2 patch')
ax.set_xticks(range(0, 9, patch_size))
ax.set_yticks(range(0, 9, patch_size))
ax.grid(color='white', linewidth=2)
ax.set_xticklabels([])
ax.set_yticklabels([])

patch_id = 0
for y in range(0, 8, patch_size):
    for x in range(0, 8, patch_size):
        ax.text(x + 0.5, y + 0.5, f'P{patch_id}', color='white', ha='center', va='center', weight='bold')
        patch_id += 1

plt.show()

## 27-3. Patch embedding

이미지 크기가 `H x W x C`이고 patch 크기가 `P x P`라면 patch 개수는 `(H / P) x (W / P)`입니다. 각 patch는 `P x P x C` 값을 가지며, 이를 펼친 뒤 linear projection으로 embedding 차원에 맞춥니다.

예제에서는 `8 x 8 x 3` 이미지를 `2 x 2` patch로 나누므로 patch는 16개입니다. patch 하나를 펼치면 길이는 `2 x 2 x 3 = 12`입니다.

In [ ]:
def image_to_patches(img, patch_size):
    h, w, c = img.shape
    patches = []
    for y in range(0, h, patch_size):
        for x in range(0, w, patch_size):
            patch = img[y:y + patch_size, x:x + patch_size, :]
            patches.append(patch.reshape(-1))
    return np.stack(patches)

patches = image_to_patches(image, patch_size)
embed_dim = 6
projection = np.random.normal(scale=0.2, size=(patches.shape[1], embed_dim))
patch_embeddings = patches @ projection

print('image shape:', image.shape)
print('patches shape:', patches.shape)
print('patch embeddings shape:', patch_embeddings.shape)

`patch embeddings shape`의 첫 번째 차원은 token 개수, 두 번째 차원은 embedding dimension입니다. 이제 이 patch embedding들은 문장의 단어 embedding처럼 Transformer에 들어갈 수 있습니다.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(patch_embeddings, cmap='viridis', aspect='auto')
ax.set_title('Patch embeddings')
ax.set_xlabel('embedding dimension')
ax.set_ylabel('patch token')
ax.set_yticks(range(16))
ax.set_yticklabels([f'P{i}' for i in range(16)])
plt.colorbar(im, ax=ax, fraction=0.046)
plt.show()

## 27-4. Class token과 positional embedding

ViT는 sequence 맨 앞에 학습 가능한 `class token`을 붙입니다. Transformer encoder를 지난 뒤 이 class token의 최종 표현을 classification head에 넣어 이미지 class를 예측합니다.

또한 patch token만 있으면 2D 위치 정보가 약하므로 positional embedding을 더합니다.

In [ ]:
class_token = np.zeros((1, embed_dim))
tokens = np.vstack([class_token, patch_embeddings])
positional_embedding = np.random.normal(scale=0.05, size=tokens.shape)
vit_input = tokens + positional_embedding

print('class token shape:', class_token.shape)
print('tokens with class token shape:', tokens.shape)
print('ViT encoder input shape:', vit_input.shape)

In [ ]:
labels = ['CLS'] + [f'P{i}' for i in range(16)]

fig, ax = plt.subplots(figsize=(10, 2.5))
ax.set_xlim(-0.5, len(labels) - 0.5)
ax.set_ylim(0, 1)
ax.axis('off')
ax.set_title('Transformer에 들어가는 token sequence')

for i, label in enumerate(labels):
    color = '#fecaca' if label == 'CLS' else '#dbeafe'
    edge = '#dc2626' if label == 'CLS' else '#2563eb'
    rect = Rectangle((i - 0.38, 0.3), 0.76, 0.4, facecolor=color, edgecolor=edge)
    ax.add_patch(rect)
    ax.text(i, 0.5, label, ha='center', va='center', fontsize=8)

plt.show()

## 27-5. 아주 단순화한 Transformer encoder 통과

실제 Transformer encoder는 multi-head attention, residual connection, layer normalization, MLP block으로 이루어집니다. 여기서는 self-attention 한 번으로 token이 서로 정보를 섞는 흐름만 확인합니다.

In [ ]:
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    exp_x = np.exp(x)
    return exp_x / exp_x.sum(axis=axis, keepdims=True)

Wq = np.random.normal(scale=0.2, size=(embed_dim, embed_dim))
Wk = np.random.normal(scale=0.2, size=(embed_dim, embed_dim))
Wv = np.random.normal(scale=0.2, size=(embed_dim, embed_dim))

Q = vit_input @ Wq
K = vit_input @ Wk
V = vit_input @ Wv

attention = softmax(Q @ K.T / np.sqrt(embed_dim), axis=1)
encoded_tokens = attention @ V
cls_representation = encoded_tokens[0]

print('attention shape:', attention.shape)
print('encoded tokens shape:', encoded_tokens.shape)
print('class representation shape:', cls_representation.shape)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
cls_attention = attention[0, 1:].reshape(4, 4)
im = ax.imshow(cls_attention, cmap='magma')
ax.set_title('CLS token이 각 patch를 참고하는 비율')
ax.set_xticks(range(4))
ax.set_yticks(range(4))
for y in range(4):
    for x in range(4):
        ax.text(x, y, f'{cls_attention[y, x]:.2f}', ha='center', va='center', color='white', fontsize=9)
plt.colorbar(im, ax=ax, fraction=0.046)
plt.show()

class token은 attention을 통해 patch token들의 정보를 모읍니다. 마지막 layer의 class token 표현은 전체 이미지 요약처럼 사용되고, classification head가 이를 class score로 바꿉니다.

In [ ]:
num_classes = 3
class_names = ['cat', 'dog', 'car']
classifier = np.random.normal(scale=0.3, size=(embed_dim, num_classes))
logits = cls_representation @ classifier
prob = softmax(logits, axis=0)

for name, p in zip(class_names, prob):
    print(f'{name}: {p:.3f}')

plt.figure(figsize=(5, 3))
plt.bar(class_names, prob, color=['#ef4444', '#3b82f6', '#22c55e'])
plt.ylim(0, 1)
plt.title('예시 classification head 출력')
plt.ylabel('probability')
plt.show()

## 27-6. CNN 분류기와 ViT 분류기의 차이

| 관점 | CNN 분류기 | ViT 분류기 |
|---|---|---|
| 입력 단위 | pixel grid와 feature map | patch token sequence |
| 핵심 연산 | convolution, pooling | self-attention, MLP |
| 이미지 요약 | global average pooling 등 | class token |
| 위치 정보 | convolution 구조에 내재 | positional embedding 추가 |
| 강점 | 지역 패턴과 데이터 효율 | 전역 관계와 scale-up |

ViT는 CNN의 모든 장점을 대체한다기보다, 충분한 데이터와 큰 모델에서 강력한 대안으로 등장했습니다. 이후 Swin Transformer처럼 CNN의 계층적 구조 아이디어를 다시 가져온 Transformer 계열도 발전했습니다.

## 정리

- ViT는 이미지를 patch로 나누고 각 patch를 token으로 취급합니다.
- patch는 flatten 후 linear projection을 거쳐 patch embedding이 됩니다.
- class token은 patch token들의 정보를 모아 최종 분류에 사용됩니다.
- positional embedding은 patch의 위치 정보를 보완합니다.
- 다음 노트북 `28_ViT_이미지_분류_실습.ipynb`에서는 사전학습된 ViT로 실제 이미지 분류 inference를 수행하는 흐름으로 넘어갑니다.